In [69]:
import numpy as np
from numba import njit
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from IPython.display import HTML
from matplotlib.animation import FuncAnimation, HTMLWriter
from mpl_toolkits.mplot3d import Axes3D      # noqa: F401
from tqdm import tqdm

In [70]:
class Planeta:

    def __init__(self, e, a, t):

        self.t = t
        self.dt = t[1] - t[0] # Paso del tiempo

        self.e = e
        self.a_ = a

        self.G = 4*np.pi**2

        self.r = np.zeros(3)
        self.v = np.zeros_like(self.r)
        self.a = np.zeros_like(self.r)

        # Iniciamos la condicion inicial


        self.r[0] = self.a_*(1-self.e)
        self.v[1] = np.sqrt(self.G*(1+self.e)/(self.a_*(1.-self.e)))


        self.R = np.zeros((len(t),len(self.r)))
        self.V = np.zeros_like(self.R)

        # El valor pasado
        self.rp = self.r

    def GetEuler(self):
        self.r += self.v*self.dt

    def GetAceleration(self):

        d = np.linalg.norm(self.r)
        self.a = -self.G/d**3 * self.r

    def Evolution(self,i):

        self.SetPosition(i)
        self.SetVelocity(i)
        self.GetAceleration()

        if i == 0:
            self.r = self.rp + self.v*self.dt

        else:

            self.rf = 2*self.r - self.rp + self.a*self.dt**2
            self.v = (self.rf - self.rp)/(2*self.dt)

            self.rp = self.r
            self.r = self.rf


    def SetPosition(self,i):
        self.R[i] = self.r

    def SetVelocity(self,i):
        self.V[i] = self.v

    def GetPosition(self,scale=1):
        return self.R[::scale]

    def GetVelocity(self,scale=1):
        return self.V[::scale]

    def GetPerihelio(self):

        Dist = np.linalg.norm(self.R,axis=1)

        timep = []

        for i in range(1,len(Dist)-1):
            if Dist[i] < Dist[i-1] and Dist[i] < Dist[i+1]:
                timep.append(self.t[i])

        return np.min(timep)

In [71]:
def GetPlanetas(t):

    Mercurio = Planeta(0.2056,0.387,t)
    Venus = Planeta(0.0067,0.7233,t)
    Tierra = Planeta(0.01671,1.,t)
    Marte = Planeta(0.0933615,1.523679,t)

    return [Mercurio,Venus,Tierra,Marte]

In [72]:
dt = 0.001
tmax = 15
t = np.arange(0.,tmax,dt)
Planetas = GetPlanetas(t)

In [73]:
def RunSimulation(t,Planetas):

    # Time evolution
    for it in tqdm(range(len(t)),desc='Running simulation', unit=' Steps' ):

        for i in range(len(Planetas)):
            Planetas[i].Evolution(it)

    return Planetas

In [74]:
Planetas = RunSimulation(t,Planetas)

Running simulation: 100%|██████████| 15000/15000 [00:01<00:00, 10878.28 Steps/s]


In [75]:
for p in Planetas:
    print(p.GetPerihelio(),p.GetPerihelio()*365.14)

0.23900000000000002 87.26846
0.552 201.55728000000002
0.97 354.1858
1.875 684.6374999999999


In [76]:
scale = 20
t1 = t[::scale]

In [77]:
fig = plt.figure(figsize=(10,5))
ax = fig.add_subplot(121)
ax1 = fig.add_subplot(122)
colors= ['r','k','b','r']

def init():
  ax.clear()
  ax.set_xlim(-2,2)
  ax.set_ylim(-2,2)

  ax1.clear()
  ax1.set_xlim(-3,3)
  ax1.set_ylim(-3,3)

def Update(i):

    init()
    ax.set_title(r'$ t=%.3f $' %(t1[i]))

    ax.scatter(0,0,s=100,color='y')
    for j, p in enumerate(Planetas):

        x = p.GetPosition(scale)[i,0]
        y = p.GetPosition(scale)[i,1]
        z = p.GetPosition(scale)[i,2]

        ax.scatter(x,y,c=colors[j],s=10)

    Mx = Planetas[1].GetPosition(scale)[:i,0] - Planetas[2].GetPosition(scale)[:i,0]
    My = Planetas[1].GetPosition(scale)[:i,1] - Planetas[2].GetPosition(scale)[:i,1]

    Marx = Planetas[3].GetPosition(scale)[:i,0] - Planetas[2].GetPosition(scale)[:i,0]
    Mary = Planetas[3].GetPosition(scale)[:i,1] - Planetas[2].GetPosition(scale)[:i,1]

    ax1.scatter(Mx,My,marker='.',label='Mercurio')
    #ax1.scatter(Marx,Mary,marker='.',label='Marte')

    return []

ani = FuncAnimation(fig, Update, frames=len(t1),
                    init_func=init, interval=50)
plt.close()

# ---------- SALIDA COMO VÍDEO HTML5 ---------
HTMLWriter.embed_limit = 40  # MB
HTML(ani.to_html5_video())